In [ ]:
# !pip install paddlepaddle-gpu paddleocr -q

In [ ]:
import cv2
import numpy as np
import subprocess
import logging
from pathlib import Path
from paddleocr import PaddleOCR
from collections import Counter

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

VIDEO_PATH    = "/content/campus_tour.mp4"
OUTPUT_PATH   = "/content/campus_tour_ocr_output.mp4"
FRAME_SKIP    = 5
CONF_THRESHOLD = 0.5

In [ ]:
logging.disable(logging.CRITICAL)
ocr = PaddleOCR(use_angle_cls=True, lang='en')
print("OCR model ready.")

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), f"Cannot open: {VIDEO_PATH}"

fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))

all_words = []
frame_idx = 0

print(f"Processing {total} frames @ {fps:.1f} fps — sampling every {FRAME_SKIP}...")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_idx % FRAME_SKIP == 0:
        results = ocr.predict(frame)
        if results and results[0]:
            rec_texts  = results[0].get('rec_texts', [])
            rec_scores = results[0].get('rec_scores', [])
            rec_polys  = results[0].get('rec_polys', [])

            for text, conf, poly in zip(rec_texts, rec_scores, rec_polys):
                if conf >= CONF_THRESHOLD and text.strip():
                    all_words.append(text.strip())
                    pts = np.array(poly, dtype=np.int32)
                    cv2.polylines(frame, [pts], True, (0, 255, 0), 2)
                    cv2.putText(frame, f"{text} ({conf:.2f})",
                                tuple(pts[0].tolist()), cv2.FONT_HERSHEY_SIMPLEX,
                                0.6, (0, 255, 0), 2)

    writer.write(frame)
    frame_idx += 1

cap.release()
writer.release()

unique = list(dict.fromkeys(all_words))
print(f"\nDone. Processed {frame_idx} frames.")
print(f"Unique words detected: {unique}")
print(f"\nAnnotated video saved → {OUTPUT_PATH}")

In [ ]:
import re
from collections import Counter

def is_valid(text):
    letters = re.sub(r'[^a-zA-Z]', '', text)
    if len(letters) < 3:               # drop short noise
        return False
    if len(text) > 1 and text == text.upper():  # all-caps is fine
        pass
    noise = len(re.sub(r'[a-zA-Z0-9 ]', '', text)) / len(text)
    if noise > 0.3:                    # too many symbols
        return False
    if re.fullmatch(r'[\d%\.\s]+', text):  # pure numbers
        return False
    return True

freq = Counter(all_words)
filtered = [w for w in unique if is_valid(w)]
# Sort by frequency — most-seen words first
filtered.sort(key=lambda w: -freq[w])

print(f"Raw: {len(unique)} → Cleaned: {len(filtered)}")
print(filtered)